# Descriptive mapping of disease counts

Computes the number of cases per 100 m population grid cell for the original dataset and every protected/masked variant, and applies the shared publication rule (at least `K_MIN` cases and `PMIN` population per cell) before export, so no map can reveal a sparsely populated, potentially identifiable cell.

**Input:** the datasets listed in `DATASETS` below, plus the population grid.
**Output:** one shapefile per dataset in `outputs_choropleth/`, ready for styling in QGIS.


In [ ]:
# Imports
import os
from pathlib import Path                      # robust path handling across platforms (Windows/Mac/Linux)
import numpy as np                            # numeric helpers (NaN, arrays)
import pandas as pd                           # tables / DataFrames
import geopandas as gpd                       # GIS vector data + spatial join


In [ ]:
#Paths
# Data directory. Override with the DATA_DIR environment variable to
# point at a different copy of the data without editing this file, e.g.
#   DATA_DIR=/path/to/data jupyter execute this_notebook.ipynb
# The data itself is not part of this repository -- see README.md.
DATA_DIR = Path(os.environ.get("DATA_DIR", "data"))

GRID_PATH = DATA_DIR / "population_grid" / "GHS_Hungary_pop_city-clipped2.shp"
GRID_ID_COL = "GID"                                 
POP_COL = "VALUE"                               
OUT_DIR = Path("outputs_choropleth")               
OUT_DIR.mkdir(parents=True, exist_ok=True)              

#Parameters
K_MIN = 5
PMIN = 50

In [ ]:
#Loading ODP and population grid

DATASETS = {                                           
    "original_points": {                          
        "type": "points",                               
        "path": DATA_DIR / "original" / "2008_eov_illness_stand.shp",     
    },
    "donut": {
        "type": "points",
        "path":DATA_DIR / "donut_masking" / "donut_points150_high900.shp",
    },
    "stat_masked": {
        "type": "points",
        "path": DATA_DIR / "statistical_masking" / "statmasked_points.shp",
    },
    "synthetic_sdv": {
        "type": "points",
        "path": DATA_DIR / "synthetic" / "synthetic_points.shp",
    },
    "areal_grid_release": {                        
        "type": "grid",
        "path": DATA_DIR / "area_aggregation" / "aggregated_points.shp",
        "layer": None,
        "count_col": "n_cases_pu",
    }
}

In [ ]:
#Load grid base (for spatial joins)
def counts_by_grid(spec):
    data = gpd.read_file(spec["path"]).to_crs(grid_base.crs)

    #count points by grid cell
    if spec["type"] == "points":
        joined = gpd.sjoin(
            data[["geometry"]],
            grid_base[[GRID_ID_COL, "geometry"]],
            how="inner",
            predicate="within"
        )

        return joined.groupby(GRID_ID_COL).size().reset_index(name="Y")

    count_col = spec["count_col"]
    data["Y"] = pd.to_numeric(data[count_col], errors="coerce").fillna(0)

    #if data already has GRID_ID_COL, skip spatial join
    if GRID_ID_COL in data.columns:
        return data.groupby(GRID_ID_COL, as_index=False)["Y"].sum()

    #spatial join to assign points to grid cells
    joined = gpd.sjoin(
        data[["Y", "geometry"]],
        grid_base[[GRID_ID_COL, "geometry"]],
        how="inner",
        predicate="intersects"
    )

    #aggregate counts by grid cell
    return joined.groupby(GRID_ID_COL, as_index=False)["Y"].sum()

In [ ]:
#Load grid base (for spatial joins)
grid = gpd.read_file(GRID_PATH)
grid[POP_COL] = pd.to_numeric(grid[POP_COL], errors="coerce").fillna(0)

#Create base grid with population for merging results
grid_base = grid[[GRID_ID_COL, POP_COL, "geometry"]].copy()

summary_rows = []

#Process each dataset
for name, spec in DATASETS.items():
    print(f"Processing {name}")

    counts = counts_by_grid(spec)

    #Merge counts with grid base
    out = grid_base.merge(counts, on=GRID_ID_COL, how="left")
    out["Y"] = out["Y"].fillna(0).astype(int)

    #Create flags for publishability
    out["k_ok"] = (out["Y"] >= K_MIN).astype(int)
    out["pop_ok"] = (out[POP_COL] >= PMIN).astype(int)
    out["publish"] = ((out["Y"] >= K_MIN) & (out[POP_COL] >= PMIN)).astype(int)

    #Value used for mapping: only publishable cells receive a count value
    out["Y_plot"] = np.where(out["publish"] == 1, out["Y"], np.nan)

    #Save results
    out.to_file(OUT_DIR / f"{name}_ddm.shp")